# Foreground Evaluation Pipeline (Multi-Class)
> Compare predicted masks from fine-tuned SAM3 model with ground truth masks per category: "pins" and "eberhard pins"

In [ ]:
#| default_exp evaluation.multi_eval_pipe

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import torch
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
import json
from sam3 import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

In [ ]:

# Base path for test data
BASE_PATH = Path('/home/hpc_storage/data/projects/userA/sam3_data/multi_test')

# Define categories with folder paths
PIN_CATEGORIES = {
    'pin': {
        'prompt': 'pin',
        'image_folder': BASE_PATH / 'images' / 'pin',
        'mask_folder': BASE_PATH / 'masks' / 'pin',
    },
    'eberhard pin': {
        'prompt': 'eberhard pin',
        'image_folder': BASE_PATH / 'images' / 'eberhard_pin',
        'mask_folder': BASE_PATH / 'masks' / 'eberhard_pin',
    },
}


# Model paths
BPE_PATH = '/home/hpc_storage/data/projects/userA/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz'
SAM3_CHECKPOINT = Path('/home/hpc_storage/data/projects/userA/sam3/models--facebook--sam3/sam3.pt')
FINETUNED_CHECKPOINT = Path('/home/hpc_storage/data/projects/userA/sam3_data/experiments/multiclass/decoder_data/I1000E200/checkpoints/checkpoint_200.pt')

# Inference settings
CONFIDENCE_THRESHOLD = 0.5

# Output directory
OUTPUT_DIR = Path('/home/hpc_storage/data/projects/userA/sam3_data/multi_test/evaluation_output')

In [ ]:
#| export
def load_model(bpe_path: str, sam3_checkpoint: Path, finetuned_checkpoint: Path, 
               device: torch.device = None) -> torch.nn.Module:
    """Load SAM3 model with fine-tuned weights."""
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    model = build_sam3_image_model(bpe_path=bpe_path, checkpoint_path=sam3_checkpoint)
    finetuned_weights = torch.load(finetuned_checkpoint, map_location=device)
    model.load_state_dict(finetuned_weights['model'])
    model.eval()
    
    print("Model loaded")
    return model

In [ ]:
# Load the model
model = load_model(
    bpe_path=BPE_PATH,
    sam3_checkpoint=SAM3_CHECKPOINT,
    finetuned_checkpoint=FINETUNED_CHECKPOINT
)

> Evaluation Metric

In [ ]:
#| export
def compute_foreground_iou(pred_mask: np.ndarray, gt_mask: np.ndarray) -> dict:
    """
    Compute Foreground IoU between predicted and ground truth masks.
    
    Foreground IoU = TP / (TP + FP + FN)
    """
    pred_mask = pred_mask.astype(bool)
    gt_mask = gt_mask.astype(bool)
    
    TP = np.logical_and(pred_mask, gt_mask).sum()
    FP = np.logical_and(pred_mask, ~gt_mask).sum()
    FN = np.logical_and(~pred_mask, gt_mask).sum()
    TN = np.logical_and(~pred_mask, ~gt_mask).sum()
    
    denominator = TP + FP + FN
    fg_iou = TP / denominator if denominator > 0 else 1.0
    
    return {
        'foreground_iou': fg_iou,
        'TP': int(TP),
        'FP': int(FP),
        'FN': int(FN),
        'TN': int(TN),
        'precision': TP / (TP + FP) if (TP + FP) > 0 else 0,
        'recall': TP / (TP + FN) if (TP + FN) > 0 else 0,
    }

In [ ]:
#| export
def load_ground_truth_mask(mask_path: Path) -> np.ndarray:
    """Load ground truth mask from file."""
    mask = np.array(Image.open(mask_path).convert('L'))
    return (mask > 127).astype(np.uint8)

In [ ]:
#| export
def get_predicted_mask(model, image: Image.Image, prompt: str, 
                       confidence_threshold: float = 0.5) -> np.ndarray:
    """Run inference and get combined predicted mask."""
    processor = Sam3Processor(model, confidence_threshold=confidence_threshold)
    inference_state = processor.set_image(image)
    processor.reset_all_prompts(inference_state)
    inference_state = processor.set_text_prompt(state=inference_state, prompt=prompt)
    
    masks = inference_state.get('masks', [])
    
    if len(masks) == 0:
        return np.zeros((image.height, image.width), dtype=np.uint8)
    
    combined_mask = np.zeros((image.height, image.width), dtype=bool)
    for mask in masks:
        if isinstance(mask, torch.Tensor):
            mask = mask.cpu().numpy()
        if mask.ndim == 3:
            mask = mask.squeeze()
        combined_mask = np.logical_or(combined_mask, mask > 0.5)
    
    return combined_mask.astype(np.uint8)

In [ ]:
#| export
def find_matching_mask(image_path: Path, masks_dir: Path) -> Path:
    """Find the corresponding ground truth mask for an image."""
    stem = image_path.stem
    for ext in ['.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp']:
        mask_path = masks_dir / f"{stem}{ext}"
        if mask_path.exists():
            return mask_path
    raise FileNotFoundError(f"No mask found for {image_path.name} in {masks_dir}")

In [ ]:
#| export
def evaluate_category(model, category_name: str, images_dir: Path, masks_dir: Path, 
                      prompt: str, confidence_threshold: float = 0.5,
                      start_idx: int = 0, end_idx: int = None,
                      max_samples: int = None) -> dict:
    """
    Evaluate model using Foreground IoU metric for a single category.
    """
    all_images = sorted(list(images_dir.glob('*.png')))
    
    if end_idx is None:
        end_idx = len(all_images)
    
    image_files = all_images[start_idx:end_idx]
    
    if max_samples:
        image_files = image_files[:max_samples]
    
    print(f"\n Evaluating '{category_name}': {len(image_files)} images")
    
    results = {
        'category': category_name,
        'prompt': prompt,
        'foreground_iou': [],
        'precision': [],
        'recall': [],
        'TP_pixels': [],
        'FP_pixels': [],
        'FN_pixels': [],
        'TN_pixels': [],
        'gt_fg_pixels': [],
        'pred_fg_pixels': [],
        'file_names': [],
        'errors': []
    }
    
    for img_path in tqdm(image_files, desc=f"Evaluating {category_name}"):
        try:
            image = Image.open(img_path).convert('RGB')
            mask_path = find_matching_mask(img_path, masks_dir)
            gt_mask = load_ground_truth_mask(mask_path)
            
            with torch.no_grad():
                pred_mask = get_predicted_mask(model, image, prompt, confidence_threshold)
            
            if pred_mask.shape != gt_mask.shape:
                pred_mask = np.array(
                    Image.fromarray(pred_mask).resize(
                        (gt_mask.shape[1], gt_mask.shape[0]), 
                        Image.NEAREST
                    )
                )
            
            fg_metrics = compute_foreground_iou(pred_mask, gt_mask)
            
            results['foreground_iou'].append(fg_metrics['foreground_iou'])
            results['precision'].append(fg_metrics['precision'])
            results['recall'].append(fg_metrics['recall'])
            results['TP_pixels'].append(fg_metrics['TP'])
            results['FP_pixels'].append(fg_metrics['FP'])
            results['FN_pixels'].append(fg_metrics['FN'])
            results['TN_pixels'].append(fg_metrics['TN'])
            results['gt_fg_pixels'].append(int(gt_mask.sum()))
            results['pred_fg_pixels'].append(int(pred_mask.sum()))
            results['file_names'].append(img_path.name)
            
        except Exception as e:
            results['errors'].append((img_path.name, str(e)))
            print(f"Error processing {img_path.name}: {e}")
    
    return results

In [ ]:
#| export
def evaluate_multiclass(model, categories: dict, confidence_threshold: float = 0.5,
                        start_idx: int = 0, end_idx: int = None,
                        max_samples: int = None) -> dict:
    """
    Evaluate model using Foreground IoU for multiple categories.
    
    Args:
        model: SAM3 model
        categories: Dictionary with category configurations
        confidence_threshold: Confidence threshold for model
        start_idx: Start index for image selection
        end_idx: End index for image selection
        max_samples: Maximum samples per category
    
    Returns:
        Dictionary with results per category
    """
    all_results = {}
    
    for cat_name, cat_config in categories.items():
        results = evaluate_category(
            model=model,
            category_name=cat_name,
            images_dir=cat_config['image_folder'],
            masks_dir=cat_config['mask_folder'],
            prompt=cat_config['prompt'],
            confidence_threshold=confidence_threshold,
            start_idx=start_idx,
            end_idx=end_idx,
            max_samples=max_samples
        )
        all_results[cat_name] = results
    
    return all_results

In [ ]:

multiclass_results = evaluate_multiclass(
    model=model,
    categories=PIN_CATEGORIES,
    confidence_threshold=CONFIDENCE_THRESHOLD,
    start_idx=0,         
    end_idx=20,          
    max_samples=None
)

In [ ]:
#| export
def calc_multiclass_summary(all_results: dict) -> dict:
    """Print and return summary statistics for multi-class Foreground IoU evaluation."""
    
    summary = {
        'categories': {}
    }
    
    # Overall totals across all categories
    grand_total_TP = 0
    grand_total_FP = 0
    grand_total_FN = 0
    
    for cat_name, results in all_results.items():
        n_samples = len(results['foreground_iou'])
        n_errors = len(results['errors'])
        
        print(f"\n CATEGORY: {cat_name.upper()} (prompt: '{results['prompt']}')")
        print(f"   Images evaluated: {n_samples} | Errors: {n_errors}")
        
        cat_summary = {
            'prompt': results['prompt'],
            'total_images': n_samples,
            'errors': n_errors,
        }
        
        if n_samples > 0:
            total_TP = sum(results['TP_pixels'])
            total_FP = sum(results['FP_pixels'])
            total_FN = sum(results['FN_pixels'])
            total_TN = sum(results['TN_pixels'])
            
            grand_total_TP += total_TP
            grand_total_FP += total_FP
            grand_total_FN += total_FN
            
            overall_fg_iou = total_TP / (total_TP + total_FP + total_FN) if (total_TP + total_FP + total_FN) > 0 else 0
            mean_fg_iou = np.mean(results['foreground_iou'])
            
            pixel_precision = total_TP / (total_TP + total_FP) if (total_TP + total_FP) > 0 else 0
            pixel_recall = total_TP / (total_TP + total_FN) if (total_TP + total_FN) > 0 else 0
            pixel_f1 = 2 * pixel_precision * pixel_recall / (pixel_precision + pixel_recall) if (pixel_precision + pixel_recall) > 0 else 0
            
            sorted_indices = np.argsort(results['foreground_iou'])
            worst_idx = sorted_indices[0]
            best_idx = sorted_indices[-1]
            
            print(f"\n FOREGROUND IoU:")
            print(f" Overall FG IoU:  {overall_fg_iou*100:6.2f}%")
            print(f" Mean FG IoU: {mean_fg_iou*100:6.2f}%")
            print(f" Precision: {pixel_precision*100:6.2f}%")
            print(f" Recall: {pixel_recall*100:6.2f}%")
            print(f" F1 Score: {pixel_f1*100:6.2f}%")
            
            print(f"\n PIXEL COUNTS:")
            print(f" TP: {total_TP:,} | FP: {total_FP:,} | FN: {total_FN:,}")
            
            print(f"\n BEST & WORST:")
            print(f"  Best:  {results['file_names'][best_idx]} (FG IoU: {results['foreground_iou'][best_idx]:.4f})")
            print(f"  Worst: {results['file_names'][worst_idx]} (FG IoU: {results['foreground_iou'][worst_idx]:.4f})")
            
            cat_summary.update({
                'overall_foreground_iou': round(overall_fg_iou * 100, 2),
                'mean_foreground_iou': round(mean_fg_iou * 100, 2),
                'pixel_precision': round(pixel_precision * 100, 2),
                'pixel_recall': round(pixel_recall * 100, 2),
                'pixel_f1': round(pixel_f1 * 100, 2),
                'total_TP_pixels': total_TP,
                'total_FP_pixels': total_FP,
                'total_FN_pixels': total_FN,
                'best_sample': {'file_name': results['file_names'][best_idx], 'fg_iou': round(results['foreground_iou'][best_idx], 4)},
                'worst_sample': {'file_name': results['file_names'][worst_idx], 'fg_iou': round(results['foreground_iou'][worst_idx], 4)},
            })
        
        summary['categories'][cat_name] = cat_summary
    
    # Grand total across all categories
    grand_fg_iou = grand_total_TP / (grand_total_TP + grand_total_FP + grand_total_FN) if (grand_total_TP + grand_total_FP + grand_total_FN) > 0 else 0
    grand_precision = grand_total_TP / (grand_total_TP + grand_total_FP) if (grand_total_TP + grand_total_FP) > 0 else 0
    grand_recall = grand_total_TP / (grand_total_TP + grand_total_FN) if (grand_total_TP + grand_total_FN) > 0 else 0
    
    print("OVERALL (ALL CATEGORIES COMBINED)")
    print(f"Foreground IoU: {grand_fg_iou*100:6.2f}%")
    print(f"Precision: {grand_precision*100:6.2f}%")
    print(f" Recall:{grand_recall*100:6.2f}%")
    
    summary['overall'] = {
        'foreground_iou': round(grand_fg_iou * 100, 2),
        'precision': round(grand_precision * 100, 2),
        'recall': round(grand_recall * 100, 2),
        'total_TP_pixels': grand_total_TP,
        'total_FP_pixels': grand_total_FP,
        'total_FN_pixels': grand_total_FN,
    }
    
    return summary

In [ ]:
# Print summary
summary = calc_multiclass_summary(multiclass_results)

In [ ]:
#| export
def save_multiclass_summary_to_json(summary: dict, all_results: dict, 
                                     output_dir: Path,
                                     filename: str = "multiclass_foreground_iou_summary.json"):
    """
    Save multi-class evaluation summary and per-image results to JSON.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / filename
    
    # Add per-image details for each category
    per_category_details = {}
    for cat_name, results in all_results.items():
        per_image_results = []
        for i in range(len(results['file_names'])):
            per_image_results.append({
                'file_name': results['file_names'][i],
                'foreground_iou': round(results['foreground_iou'][i], 4),
                'precision': round(results['precision'][i], 4),
                'recall': round(results['recall'][i], 4),
                'TP_pixels': results['TP_pixels'][i],
                'FP_pixels': results['FP_pixels'][i],
                'FN_pixels': results['FN_pixels'][i],
            })
        per_category_details[cat_name] = {
            'per_image_results': per_image_results,
            'errors': results['errors']
        }
    
    output_data = {
        'summary': summary,
        'per_category_details': per_category_details
    }
    
    with open(output_path, 'w') as f:
        json.dump(output_data, f, indent=2)
    
    return output_path

In [ ]:
# Save the summary
save_multiclass_summary_to_json(summary, multiclass_results, output_dir=OUTPUT_DIR)

In [ ]:
#| export
if __name__ == '__main__':
    # Base path for test data
    BASE_PATH = Path('/home/hpc_storage/data/projects/userA/sam3_data/multi_test')
    PIN_CATEGORIES = {
        'pin': {
            'prompt': 'pin',
            'image_folder': BASE_PATH / 'images' / 'pin',
            'mask_folder': BASE_PATH / 'masks' / 'pin',
        },
        'eberhard pin': {
            'prompt': 'eberhard pin',
            'image_folder': BASE_PATH / 'images' / 'eberhard_pin',
            'mask_folder': BASE_PATH / 'masks' / 'eberhard_pin',
        },
    }
    OUTPUT_DIR = Path('/home/hpc_storage/data/projects/userA/sam3_data/multi_test/evaluation_output')

    BPE_PATH = '/home/hpc_storage/data/projects/userA/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz'
    SAM3_CHECKPOINT = Path('/home/hpc_storage/data/projects/userA/sam3/models--facebook--sam3/sam3.pt')
    FINETUNED_CHECKPOINT = Path('/home/hpc_storage/data/projects/userA/sam3_data/experiments/multiclass/decoder_data/I1000E200/checkpoints/checkpoint_200.pt')
    CONFIDENCE_THRESHOLD = 0.5


    # Load the model
    model = load_model(
        bpe_path=BPE_PATH,
        sam3_checkpoint=SAM3_CHECKPOINT,
        finetuned_checkpoint=FINETUNED_CHECKPOINT
    )

    multiclass_results = evaluate_multiclass(
        model=model,
        categories=PIN_CATEGORIES,
        confidence_threshold=CONFIDENCE_THRESHOLD,
        start_idx=0,         
        end_idx=20,          
        max_samples=None
    )

    #calculate summary
    summary = calc_multiclass_summary(multiclass_results)
    # Save summary
    save_multiclass_summary_to_json(summary, multiclass_results)

In [ ]:
# visualize single image

def visualize_category_sample(category_name: str, image_path: str, model, 
                               prompt: str, masks_dir: Path,
                               confidence_threshold: float = 0.5):
    """Visualize a single image with Foreground IoU metrics."""
    image_path = Path(image_path)
    
    image = Image.open(image_path).convert('RGB')
    mask_path = find_matching_mask(image_path, masks_dir)
    gt_mask = load_ground_truth_mask(mask_path)
    
    with torch.no_grad():
        pred_mask = get_predicted_mask(model, image, prompt, confidence_threshold)
    
    if pred_mask.shape != gt_mask.shape:
        pred_mask = np.array(
            Image.fromarray(pred_mask).resize(
                (gt_mask.shape[1], gt_mask.shape[0]), Image.NEAREST
            )
        )
    
    fg_metrics = compute_foreground_iou(pred_mask, gt_mask)
    
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    
    axes[0].imshow(image)
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    axes[1].imshow(gt_mask, cmap='gray')
    axes[1].set_title(f'Ground Truth\n({gt_mask.sum():,} fg pixels)')
    axes[1].axis('off')
    
    axes[2].imshow(pred_mask, cmap='gray')
    axes[2].set_title(f'Predicted\n({pred_mask.sum():,} fg pixels)')
    axes[2].axis('off')
    
    overlay = np.zeros((*gt_mask.shape, 3), dtype=np.uint8)
    overlay[np.logical_and(pred_mask, gt_mask)] = [0, 255, 0]
    overlay[np.logical_and(pred_mask, ~gt_mask.astype(bool))] = [255, 0, 0]
    overlay[np.logical_and(~pred_mask.astype(bool), gt_mask)] = [0, 0, 255]
    
    axes[3].imshow(overlay)
    axes[3].set_title(f'FG IoU: {fg_metrics["foreground_iou"]:.3f}')
    axes[3].axis('off')
    
    plt.suptitle(f'{category_name}: {image_path.name}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f" TP: {fg_metrics['TP']:,} | FP: {fg_metrics['FP']:,} | FN: {fg_metrics['FN']:,}")
    print(f"  FG IoU: {fg_metrics['foreground_iou']:.4f} | Precision: {fg_metrics['precision']:.4f} | Recall: {fg_metrics['recall']:.4f}")


In [ ]:
# Visualize best and worst for each category
for cat_name, results in multiclass_results.items():
    if len(results['foreground_iou']) > 0:
        cat_config = PIN_CATEGORIES[cat_name]
        sorted_indices = np.argsort(results['foreground_iou'])
        
        print(f"CATEGORY: {cat_name.upper()}")
        
        print(f"\nWORST performing sample:")
        worst_idx = sorted_indices[0]
        visualize_category_sample(
            cat_name, 
            cat_config['image_folder'] / results['file_names'][worst_idx],
            model, cat_config['prompt'], cat_config['mask_folder']
        )
        
        print(f"\nBEST performing sample:")
        best_idx = sorted_indices[-1]
        visualize_category_sample(
            cat_name,
            cat_config['image_folder'] / results['file_names'][best_idx],
            model, cat_config['prompt'], cat_config['mask_folder']
        )

In [ ]:
# visualize any image by name

def visualize_image(image_name: str, category: str = 'pin'):
    """
    Visualize any image by its filename.
    
    Args:
        image_name: Name of the image file (e.g., 'image_001.png')
        category: Category name ('pin' or 'eberhard pin')
    """
    cat_config = PIN_CATEGORIES[category]
    image_path = cat_config['image_folder'] / image_name
    
    visualize_category_sample(
        category_name=category,
        image_path=image_path,
        model=model,
        prompt=cat_config['prompt'],
        masks_dir=cat_config['mask_folder'],
        confidence_threshold=CONFIDENCE_THRESHOLD
    )

visualize_image("5_9_1_2024103175118431_0.png", category='eberhard pin')

In [ ]:
CURRENT_NB='/home/hpc_storage/data/projects/userA/sam3_sy/nbs'
import os
os.chdir(CURRENT_NB)

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
def foo(): pass

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()